In [ ]:
import pandas as pd
import numpy as np
import os

GT_PATH = "/home/hp/VTAIRACE/source_lam/dataset/Train/Public_train.csv"
OUTPUT_CSV = "Submission3D.csv"

print("Bắt đầu đánh giá (MCE + OE)...")

if os.path.exists(OUTPUT_CSV) and os.path.exists(GT_PATH):
    gt = pd.read_csv(GT_PATH)
    pred = pd.read_csv(OUTPUT_CSV)

    # Khớp tên file
    pred['image_filename'] = pred['image_filename'].apply(
         lambda x: x if x.startswith('image_') else f"image_{x}"
    )

    merged = pd.merge(gt, pred, on='image_filename', suffixes=('_gt', '_pred'))

    # Loại bỏ bất kỳ hàng nào dự đoán thiếu (dù là tâm hay góc)
    merged = merged.dropna(subset=['x_pred', 'y_pred', 'z_pred', 'Rx_pred', 'Ry_pred', 'Rz_pred'])

    if len(merged) > 0:

        # ===================================================
        # 1. TÍNH TOÁN MCE 
        # ===================================================
        merged['err'] = np.sqrt(
            (merged['x_gt'] - merged['x_pred'])**2 +
            (merged['y_gt'] - merged['y_pred'])**2 +
            (merged['z_gt'] - merged['z_pred'])**2
        )
        merged['MCE_i'] = np.minimum(merged['err'] / 0.05, 1.0)
        MCE = merged['MCE_i'].mean()

        # ===================================================
        # 2. TÍNH TOÁN OE 
        # ===================================================

        # Lấy vector pháp tuyến n (GT) và n_hat (Pred)
        gt_normals = merged[['Rx_gt', 'Ry_gt', 'Rz_gt']].values
        pred_normals = merged[['Rx_pred', 'Ry_pred', 'Rz_pred']].values

        # Đảm bảo chúng là vector đơn vị (an toàn)
        gt_norm = np.linalg.norm(gt_normals, axis=1, keepdims=True) + 1e-8
        pred_norm = np.linalg.norm(pred_normals, axis=1, keepdims=True) + 1e-8
        gt_normals_unit = gt_normals / gt_norm
        pred_normals_unit = pred_normals / pred_norm

        # Tính tích vô hướng (dot product)
        dot_product = np.einsum('ij,ij->i', gt_normals_unit, pred_normals_unit)

        # Kẹp giá trị trong [-1, 1] để tránh lỗi số học
        dot_product = np.clip(dot_product, -1.0, 1.0)

        # Tính góc theta_OE (bằng độ)
        merged['theta_oe'] = np.degrees(np.arccos(dot_product))

        # Chuẩn hóa OE_i
        THETA_MAX = 20.0 # Ngưỡng 20 độ
        merged['OE_i'] = np.minimum(merged['theta_oe'] / THETA_MAX, 1.0)

        # Tính OE trung bình
        OE = merged['OE_i'].mean()

        # ===================================================
        # 3. TÍNH ĐIỂM AC 
        # ===================================================
        score_mce_part = (1 - MCE) * 0.7 # 70%
        score_oe_part = (1 - OE) * 0.3 # 30%
        AC_score = ((score_mce_part + score_oe_part) ** 4) * 100 

        print(f"\n{'='*50}")
        print(f"ĐÁNH GIÁ KẾT QUẢ PS6D (trên tập train)")
        print(f"{'='*50}")
        print(f"Mean Center Error (MCE): {MCE:.4f}")
        print(f"Mean Orientation Error (OE): {OE:.4f}")
        print(f"--- ĐIỂM SỐ CUỐI CÙNG (AC) ---: {AC_score:.2f} điểm")

        print(f"\n--- Thống kê MCE ---")
        print(f"  Trung vị sai số (Tâm): {merged['err'].median()*1000:.2f} mm")
        print(f"  Trung bình sai số (Tâm): {merged['err'].mean()*1000:.2f} mm")
        print(f"  % ảnh hợp lệ (Tâm ≤ 5cm): {(merged['err'] <= 0.05).mean()*100:.2f}%")

        print(f"\n--- Thống kê OE ---")
        print(f"  Trung vị sai số (Góc): {merged['theta_oe'].median():.2f} độ")
        print(f"  Trung bình sai số (Góc): {merged['theta_oe'].mean():.2f} độ")
        print(f"  % ảnh hợp lệ (Góc ≤ 20°): {(merged['theta_oe'] <= THETA_MAX).mean()*100:.2f}%")

    else:
        print("\nKhông có dự đoán hợp lệ nào để đánh giá (có thể do tên file không khớp hoặc dự đoán NaN).")
else:
    print(f"Lỗi: Không tìm thấy file {OUTPUT_CSV} hoặc {GT_PATH}.")